**Export the Name-Search Result Tables**

This notebook gathers the results of the name-search experiments into CSV tables from which the results report is written. Most results are CSV files already (`./results/metrics.csv`, the per-query files, `./results/k4_coefficients.csv`, `./results/exp4_test_metrics.csv` and `./results/exp4_failures.csv`); this notebook adds a table for every result that exists only as JSON, and one summary row per configuration. It reads `./results/`, `./eval/profile.json` and the evaluation files, changes none of them, and runs no experiment. It outputs, in `./results/tables/`:

1. `data_profile.csv` and `given_name_relation.csv`: the profile of the current transcription beside the figures the experiment design quotes from the older extraction, and the Spanish–Hebrew given-name relation (the design's Figure 1)
2. `evaluation_set.csv`: the persons per split and fold and the queries per set and split
3. `preflight.csv`: each asset's status and sanity check
4. `configurations.csv` and `qs3_qs4_subsets.csv`: one row per development configuration (its settings, tuned parameters, metrics with intervals, latency, place in the decision rule and role), and the given-name and surname subsets of QS3 and QS4
5. `freeze_decision.csv`: the decision rule's outcome and the frozen parameters of the selected configuration, the runner-up and the reference
6. `exp4_mcnemar.csv`, `exp4_latency.csv` and `exp4_failure_causes.csv`: the McNemar tests, the cold and warm latency, and the error analysis counts of Experiment 4

**1. Configuration and Results**

This cell defines the paths and the figures the design quotes, and reads every result file.

1. `DESIGN_FIGURES`: the counts the experiment design (§2 and Figure 1) reports for the older extraction, `extracted_text_old.csv`, set beside the current file's in `data_profile.csv`
2. `dev_row` and `test_row`: look-ups of the development metrics (mean over the five folds) and of the test metrics

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and results  —  every result file, read once
#
# The tables only reorganise numbers the experiments wrote; nothing here is
# recomputed from the data, so a table can be traced to its source file.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
from collections import Counter
from pathlib import Path

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
RESULTS_DIR = Path("results")                  # every result file
EVAL_DIR    = Path("eval")                     # the evaluation set
TABLES_DIR  = RESULTS_DIR / "tables"           # this notebook's output
QUERY_SETS  = ("QS1", "QS2", "QS3", "QS4", "QS5")
RULE        = "─" * 72                         # frames every printed report

# The design's figures for the older extraction (experiment_design.pdf, §2).
DESIGN_FIGURES = {
    "rows": 828, "graves": 794, "rows_with_given_and_surname": 668, "eligible_rows": 657,
    "named_rows_full_name_latin": 587, "named_rows_full_name_hebrew": 444,
    "named_rows_full_name_both": 366, "named_rows_full_name_latin_only": 221,
    "named_rows_full_name_hebrew_only": 78, "women_latin_surnames": 513,
    "women_latin_surnames_with_de": 411, "hebrew_surnames": 448,
    "hebrew_surnames_yiddish_orthography": 188, "hebrew_given_names": 583,
    "hebrew_given_names_double": 127,
    "relation_rows": 473, "relation_phonetic": 182, "relation_initial": 191,
    "relation_neither": 100,
}


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(name: str, rows: list[dict]) -> Path:
    """Write `rows` to TABLES_DIR/<name>.csv; return the path."""
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    path = TABLES_DIR / f"{name}.csv"
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    return path


def read_json(path: Path):
    """Read the JSON file `path`."""
    return json.loads(path.read_text(encoding="utf-8"))


metric_rows = read_csv(RESULTS_DIR / "metrics.csv")
test_rows = read_csv(RESULTS_DIR / "exp4_test_metrics.csv")
frozen = read_json(RESULTS_DIR / "frozen_config.json")
profile = read_json(EVAL_DIR / "profile.json")


def dev_row(config_id: str, query_set: str = "macro", subset: str = "all") -> dict:
    """Return the development row (mean over the five folds) of one configuration."""
    return next(r for r in metric_rows if r["config_id"] == config_id and r["split"] == "dev"
                and r["fold"] == "mean" and r["query_set"] == query_set
                and r["subset"] == subset)


def test_row(role: str, stratum: str = "overall", value: str = "macro") -> dict:
    """Return one test row of Experiment 4 (role: selected, runner_up or reference)."""
    return next(r for r in test_rows if r["role"] == role and r["stratum"] == stratum
                and r["value"] == value)


dev_ids = sorted({r["config_id"] for r in metric_rows
                  if r["split"] == "dev" and r["status"] == "ok"})
written = {}
print(f"✓ Configuration and results loaded ({len(dev_ids)} development configurations, "
      f"{len(test_rows)} test rows).")

✓ Configuration and results loaded (35 development configurations, 42 test rows).


**2. Data, Evaluation Set and Preflight Tables**

This cell takes `profile`, the evaluation files and `preflight.json` and writes:

1. `data_profile.csv`: each count of the current transcription beside the design's figure for the older extraction, with the difference
2. `given_name_relation.csv`: the three classes of the design's Figure 1 with their counts, shares and examples
3. `evaluation_set.csv`: the persons per split and fold, and the queries per set and split
4. `preflight.csv`: each asset's purpose, status and sanity-check details

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Data, evaluation set and preflight tables
# ─────────────────────────────────────────────────────────────────────────────
if "written" not in globals():
    raise RuntimeError("Run the 'Configuration and Results' cell first.")

_relation = profile["given_name_relation"]
_current = {k: v for k, v in profile.items() if isinstance(v, int)}
_current["named_rows_full_name_latin_only"] = (profile["named_rows_full_name_latin"]
                                               - profile["named_rows_full_name_both"])
_current["named_rows_full_name_hebrew_only"] = (profile["named_rows_full_name_hebrew"]
                                                - profile["named_rows_full_name_both"])
_current |= {f"relation_{k}": _relation[k] for k in ("rows", "phonetic", "initial", "neither")}
_current |= {f"persons_{k}": v for k, v in profile["persons_by_coverage"].items()}
written["data_profile"] = write_csv("data_profile", [
    {"metric": name, "current_file": value, "design_old_file": DESIGN_FIGURES.get(name, ""),
     "difference": value - DESIGN_FIGURES[name] if name in DESIGN_FIGURES else ""}
    for name, value in _current.items()])
written["given_name_relation"] = write_csv("given_name_relation", [
    {"class": name, "rows": _relation[name], "share": round(_relation[name] / _relation["rows"], 4),
     "examples": "; ".join(_relation["examples"][name])}
    for name in ("phonetic", "initial", "neither")])

_persons = [*read_csv(EVAL_DIR / "persons_dev.csv"), *read_csv(EVAL_DIR / "persons_test.csv")]
_queries = [*read_csv(EVAL_DIR / "queries_dev.csv"), *read_csv(EVAL_DIR / "queries_test.csv")]
_person_counts = Counter((p["split"], p["fold"] or "all") for p in _persons)
_query_counts = Counter((q["split"], q["set"]) for q in _queries)
written["evaluation_set"] = write_csv("evaluation_set", [
    *({"item": "persons", "split": split, "group": f"fold {fold}" if fold != "all" else "all",
       "count": count} for (split, fold), count in sorted(_person_counts.items())),
    *({"item": "queries", "split": split, "group": query_set, "count": count}
      for (split, query_set), count in sorted(_query_counts.items()))])
written["preflight"] = write_csv("preflight", [
    {"asset": asset, "purpose": entry["purpose"], "status": entry["status"],
     "details": json.dumps({k: v for k, v in entry.items() if k not in ("purpose", "status")},
                           ensure_ascii=False)}
    for asset, entry in read_json(RESULTS_DIR / "preflight.json").items()])

print(f"{RULE}\nData, evaluation set and preflight tables\n{RULE}")
for _name in ("data_profile", "given_name_relation", "evaluation_set", "preflight"):
    print(f"  OK {_name + '.csv':<26} {len(read_csv(written[_name])):>4} rows")
print(f"{RULE}\n  {len(_persons)} persons and {len(_queries)} queries tabulated")

────────────────────────────────────────────────────────────────────────
Data, evaluation set and preflight tables
────────────────────────────────────────────────────────────────────────
  OK data_profile.csv             24 rows
  OK given_name_relation.csv       3 rows
  OK evaluation_set.csv           16 rows
  OK preflight.csv                 9 rows
────────────────────────────────────────────────────────────────────────
  637 persons and 2158 queries tabulated


**3. Configuration and Decision Tables**

This cell takes the development rows of `metrics.csv` and `frozen_config.json` and writes:

1. `configurations.csv`: one row per configuration of Experiments 1–3 with its settings, the parameters tuned on all five folds, the fold-mean macro R@10 and MRR with their 95 % intervals, unrelated@10, the empty rate, latency, R@10 per query set, whether it was carried forward, its simplicity level, whether it lies within the best configuration's interval and under the latency budget, and its role (`selected`, `runner_up`, `reference` or empty)
2. `qs3_qs4_subsets.csv`: the form-level given-name and surname R@10 and MRR of QS3 and QS4 for every configuration measured on them
3. `freeze_decision.csv`: the decision rule, the best configuration's interval, and the frozen configurations with their parameters and hashes

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and decision tables  —  development results and the freeze
# ─────────────────────────────────────────────────────────────────────────────
if "written" not in globals():
    raise RuntimeError("Run the 'Configuration and Results' cell first.")

ROLES = {frozen[role]["spec"]["config_id"]: role for role in ("selected", "runner_up", "reference")}
_decision = {row["config_id"]: row for row in frozen["decision_table"]}


def configuration_row(config_id: str) -> dict:
    """Return the summary row of one development configuration."""
    row, decision = dev_row(config_id), _decision[config_id]
    settings = {k: row[k] for k in ("experiment", "config_id", "generator", "dictionary",
                                    "placement", "vocab", "retrieval", "ranker", "aggregation")}
    metrics = {k: row[k] for k in ("params", "n_queries", "r_at_10", "r_at_10_lo", "r_at_10_hi",
                                   "mrr", "mrr_lo", "mrr_hi", "unrelated_at_10", "empty_rate",
                                   "latency_p50_ms", "latency_p95_ms")}
    per_set = {f"r_at_10_{s}": dev_row(config_id, s)["r_at_10"] for s in QUERY_SETS}
    return {**settings, **metrics, **per_set, "carried_forward": row["carried_forward"],
            "simplicity_level": decision["level"], "components": decision["components"],
            "within_ci_of_best": decision["within_ci_of_best"],
            "under_latency_budget": decision["under_latency_budget"],
            "role": ROLES.get(config_id, ""), "notes": row["notes"]}


written["configurations"] = write_csv("configurations", sorted(
    (configuration_row(c) for c in dev_ids), key=lambda r: -float(r["r_at_10"])))
written["qs3_qs4_subsets"] = write_csv("qs3_qs4_subsets", [
    {"config_id": r["config_id"], "query_set": r["query_set"], "subset": r["subset"],
     "n_queries": r["n_queries"], "r_at_10": r["r_at_10"], "mrr": r["mrr"]}
    for r in metric_rows if r["split"] == "dev" and r["fold"] == "mean"
    and r["subset"] in ("given", "surname")])
written["freeze_decision"] = write_csv("freeze_decision", [
    {"role": "rule", "config_id": frozen["best"], "config_hash": "",
     "params": json.dumps({"rule": frozen["rule"], "best_primary_ci": frozen["best_primary_ci"]}),
     "component_params": ""},
    *({"role": role, "config_id": frozen[role]["spec"]["config_id"],
       "config_hash": frozen[role]["config_hash"], "params": json.dumps(frozen[role]["params"]),
       "component_params": json.dumps(frozen[role].get("component_params", {}))}
      for role in ("selected", "runner_up", "reference"))])

print(f"{RULE}\nConfiguration and decision tables\n{RULE}")
for _row in read_csv(written["configurations"])[:6]:
    print(f"  {_row['config_id']:<36} R@10 {float(_row['r_at_10']):.3f}  "
          f"level {_row['simplicity_level']}  {_row['role']}")
print(f"{RULE}\n  {len(dev_ids)} configurations; "
      f"selected {frozen['selected']['spec']['config_id']}")

────────────────────────────────────────────────────────────────────────
Configuration and decision tables
────────────────────────────────────────────────────────────────────────
  exp3_K2_sum                          R@10 0.905  level 4  selected
  exp3_K2_product                      R@10 0.904  level 4  runner_up
  exp3_K4_sum                          R@10 0.886  level 5  
  exp2_P3_E4_D3_archive_dict_cutoff    R@10 0.885  level 3  
  exp3_K4_product                      R@10 0.882  level 5  
  exp3_K3_product                      R@10 0.878  level 4  
────────────────────────────────────────────────────────────────────────
  35 configurations; selected exp3_K2_sum


**4. Experiment 4 Tables**

This cell takes the JSON outputs of `step_7_confirm_on_test_split.ipynb` and its failure list and writes:

1. `exp4_mcnemar.csv`: each comparison of the selected configuration's hits with another's, the discordant counts and the exact p-value
2. `exp4_latency.csv`: each configuration's cold start (loading the models and index, then the first query) and its warm median and 95th-percentile latency
3. `exp4_failure_causes.csv`: the failed QS3 and QS4 queries per heuristic cause, with the share of all cross-script test queries

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 4 tables  —  McNemar tests, latency and failure causes
# ─────────────────────────────────────────────────────────────────────────────
if "written" not in globals():
    raise RuntimeError("Run the 'Configuration and Results' cell first.")

_mcnemar = read_json(RESULTS_DIR / "exp4_mcnemar.json")
_latency = read_json(RESULTS_DIR / "exp4_latency.json")
_failures = read_csv(RESULTS_DIR / "exp4_failures.csv")
_cross_script = sum(int(test_row("selected", "query_set", s)["n_queries"])
                    for s in ("QS3", "QS4"))
written["exp4_mcnemar"] = write_csv("exp4_mcnemar", [
    {"comparison": f"selected vs {other}",
     "selected_config": frozen["selected"]["spec"]["config_id"],
     "other_config": frozen[other]["spec"]["config_id"],
     "selected_only_hits": test["selected_only"],
     "other_only_hits": test["other_only"],
     "discordant": test["selected_only"] + test["other_only"], "exact_p_value": test["p_value"]}
    for other, test in _mcnemar.items()])
written["exp4_latency"] = write_csv("exp4_latency", [
    {"role": role, "config_id": frozen[role]["spec"]["config_id"],
     "cold_load_ms": round(_latency[role]["cold"]["load_ms"], 2),
     "cold_first_query_ms": round(_latency[role]["cold"]["first_query_ms"], 2),
     "warm_p50_ms": round(_latency[role]["warm_p50_ms"], 2),
     "warm_p95_ms": round(_latency[role]["warm_p95_ms"], 2)}
    for role in ("selected", "runner_up", "reference")])
written["exp4_failure_causes"] = write_csv("exp4_failure_causes", [
    {"cause": cause, "failed_queries": count,
     "share_of_failures": round(count / len(_failures), 4),
     "share_of_cross_script_queries": round(count / _cross_script, 4)}
    for cause, count in Counter(f["cause"] for f in _failures).most_common()])

print(f"{RULE}\nExperiment 4 tables\n{RULE}")
for _row in read_csv(written["exp4_mcnemar"]):
    print(f"  {_row['comparison']:<24} {_row['selected_only_hits']:>3} vs "
          f"{_row['other_only_hits']:>3}   p = {float(_row['exact_p_value']):.3g}")
for _row in read_csv(written["exp4_failure_causes"]):
    print(f"  {_row['cause']:<24} {_row['failed_queries']:>3} failures")
print(f"{RULE}\n  {len(_failures)} of {_cross_script} cross-script test queries failed")

────────────────────────────────────────────────────────────────────────
Experiment 4 tables
────────────────────────────────────────────────────────────────────────
  selected vs runner_up      0 vs   1   p = 1
  selected vs reference    159 vs   1   p = 2.2e-46
  ranking_error             24 failures
  transcription_error        6 failures
  missing_equivalence        1 failures
────────────────────────────────────────────────────────────────────────
  31 of 128 cross-script test queries failed


**5. Consistency Check**

This cell reads every table back from disk and checks it against the result files it came from, independently of the cells above:

1. every table exists and has at least one row
2. `configurations.csv` holds every development configuration of `metrics.csv` once, with the same fold-mean macro R@10, and exactly one each of the roles `selected`, `runner_up` and `reference`, matching `frozen_config.json`
3. the McNemar counts and the failure counts add up to the totals of `exp4_mcnemar.json` and `exp4_failures.csv`

It ends with the tables written and the result of the check.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the tables against their source files
# ─────────────────────────────────────────────────────────────────────────────
if "exp4_failure_causes" not in written:
    raise RuntimeError("Run every cell above first.")


def table_problems() -> list[str]:
    """Re-read the tables and check them against the result files."""
    problems = [f"{name}.csv is empty" for name, path in written.items() if not read_csv(path)]
    rows = read_csv(TABLES_DIR / "configurations.csv")
    expected = {r["config_id"]: r["r_at_10"] for r in read_csv(RESULTS_DIR / "metrics.csv")
                if r["split"] == "dev" and r["fold"] == "mean" and r["query_set"] == "macro"
                and r["subset"] == "all"}
    if sorted(r["config_id"] for r in rows) != sorted(expected):
        problems.append("configurations.csv does not hold each configuration once")
    problems += [f"{r['config_id']}: R@10 differs" for r in rows
                 if r["r_at_10"] != expected.get(r["config_id"])]
    stored = read_json(RESULTS_DIR / "frozen_config.json")
    for role in ("selected", "runner_up", "reference"):
        holders = [r["config_id"] for r in rows if r["role"] == role]
        if holders != [stored[role]["spec"]["config_id"]]:
            problems.append(f"role {role} held by {holders}")
    mcnemar_rows = read_csv(TABLES_DIR / "exp4_mcnemar.csv")
    source = read_json(RESULTS_DIR / "exp4_mcnemar.json")
    if sum(int(r["discordant"]) for r in mcnemar_rows) != sum(
            v["selected_only"] + v["other_only"] for v in source.values()):
        problems.append("McNemar counts differ from exp4_mcnemar.json")
    causes = read_csv(TABLES_DIR / "exp4_failure_causes.csv")
    if sum(int(r["failed_queries"]) for r in causes) != len(
            read_csv(RESULTS_DIR / "exp4_failures.csv")):
        problems.append("failure counts differ from exp4_failures.csv")
    return problems


_problems = table_problems()
print(f"{RULE}\nConsistency: {TABLES_DIR}/ against {RESULTS_DIR}/\n{RULE}")
for _name, _path in written.items():
    print(f"  OK {_path.name:<28} {len(read_csv(_path)):>4} rows")
for _problem in _problems:
    print(f"  !! {_problem}")
print(f"{RULE}\n  {len(written)} tables written  "
      f"({'consistent' if not _problems else f'{len(_problems)} problem(s)'})")
print("\n✓ Result tables complete (CSV tables for the results report).")
print(f"  tables consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: results/tables/ against results/
────────────────────────────────────────────────────────────────────────
  OK data_profile.csv               24 rows
  OK given_name_relation.csv         3 rows
  OK evaluation_set.csv             16 rows
  OK preflight.csv                   9 rows
  OK configurations.csv             35 rows
  OK qs3_qs4_subsets.csv           108 rows
  OK freeze_decision.csv             4 rows
  OK exp4_mcnemar.csv                2 rows
  OK exp4_latency.csv                3 rows
  OK exp4_failure_causes.csv         3 rows
────────────────────────────────────────────────────────────────────────
  10 tables written  (consistent)

✓ Result tables complete (CSV tables for the results report).
  tables consistent: True
